# week 5 - pyspark fundamentals


## dataframes and immutability


## 1. spark session setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, count, avg, min as spark_min, max as spark_max, trim, when

spark = SparkSession.builder.appName("week4_superstore_analysis").getOrCreate()

print("spark session created")
print("spark version:", spark.version)

## 2. loading the dataset

In [ ]:
csv_path = "Superstore.csv"

df = spark.read.csv(csv_path, header=True, inferSchema=True)

print("rows loaded:", df.count())
print("columns:", len(df.columns))

## 3. exploring the schema

In [ ]:
df.printSchema()
df.show(5)

## 4. data cleaning - duplicates

In [ ]:
total_rows = df.count()
distinct_rows = df.distinct().count()
print("total rows:", total_rows)
print("distinct rows:", distinct_rows)
print("duplicate rows:", total_rows - distinct_rows)

df_clean = df.dropDuplicates()
print("rows after removing duplicates:", df_clean.count())

## 5. data cleaning - null values

In [ ]:
null_counts = df_clean.select([
    spark_sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns
])
null_counts.show()

In [ ]:
# postal code nulls are a small number of rows so just drop them
df_clean = df_clean.dropna(subset=["Postal Code"])

# customer name has a few nulls, fill those instead of dropping
df_clean = df_clean.fillna({"Customer Name": "unknown"})

print("rows after handling nulls:", df_clean.count())

## 6. filtering conditions
note: this dataset doesnt have an age column, so using the sales amount as the numeric range filter instead, along with category and region.

In [ ]:
filtered_df = df_clean.filter(
    (col("Sales") >= 50) & (col("Sales") <= 500) &
    (col("Category") == "Furniture") &
    (col("Region") == "West")
)

print("rows after filtering:", filtered_df.count())
filtered_df.select("Order ID", "Category", "Region", "Sales").show(10)

## 7. aggregation functions

In [ ]:
df_clean.select(
    count("*").alias("total_orders"),
    spark_sum("Sales").alias("total_sales"),
    avg("Sales").alias("avg_sales"),
    spark_min("Sales").alias("min_sales"),
    spark_max("Sales").alias("max_sales")
).show()

## 8. groupby and conditions on aggregated results

In [ ]:
category_summary = df_clean.groupBy("Category").agg(
    spark_sum("Sales").alias("total_sales"),
    spark_sum("Profit").alias("total_profit"),
    count("*").alias("order_count")
)
category_summary.show()

In [ ]:
# only keep categories where total sales is above 20000
high_sales_categories = category_summary.filter(col("total_sales") > 20000)
print("categories with total sales above 20000:")
high_sales_categories.show()

In [ ]:
# group by two columns together
region_category_summary = df_clean.groupBy("Region", "Category").agg(
    spark_sum("Sales").alias("total_sales"),
    avg("Profit").alias("avg_profit")
).orderBy(col("total_sales").desc())

region_category_summary.show()

## 9. wide transformations and shuffle
groupBy, orderBy, join and distinct are wide transformations, they need data from other partitions to produce each output row (e.g. to group all rows with the same category together), so spark shuffles data across the network to make that happen. filter and select are narrow transformations, they only look at the data already in the same partition, so they dont need a shuffle and are much cheaper to run. shuffles are usually the slowest part of a spark job, which is why minimizing groupBy/join calls (or using broadcast joins for small lookup tables) matters for performance.

## 10. schema changes - casting and renaming

In [ ]:
df_clean = df_clean.withColumnRenamed("Postal Code", "PostalCode")

df_clean = df_clean.withColumn("Sales", col("Sales").cast("double"))
df_clean = df_clean.withColumn("Profit", col("Profit").cast("double"))

df_clean.printSchema()

## 11. handling inconsistent data
some rows had extra whitespace instead of an actual null in the region column, this cleans that up before dropping anything.

In [ ]:
df_clean = df_clean.withColumn("Category", trim(col("Category")))
df_clean = df_clean.withColumn("Region", trim(col("Region")))

# turn empty strings into actual nulls so dropna can catch them
df_clean = df_clean.withColumn(
    "Region",
    when(col("Region") == "", None).otherwise(col("Region"))
)

print("rows before dropping inconsistent region/category:", df_clean.count())
df_clean = df_clean.dropna(subset=["Category", "Region"])
print("rows after cleaning inconsistent data:", df_clean.count())

## 12. full pipeline
putting all the cleaning and aggregation steps together into one function, so the whole thing can run on the raw csv in one call.

In [ ]:
def process_superstore_data(path):
    df = spark.read.csv(path, header=True, inferSchema=True)

    df = df.dropDuplicates()
    df = df.dropna(subset=["Postal Code"])
    df = df.fillna({"Customer Name": "unknown"})

    df = df.withColumn("Category", trim(col("Category")))
    df = df.withColumn("Region", trim(col("Region")))
    df = df.withColumn("Region", when(col("Region") == "", None).otherwise(col("Region")))
    df = df.dropna(subset=["Category", "Region"])

    df = df.withColumn("Sales", col("Sales").cast("double"))
    df = df.withColumn("Profit", col("Profit").cast("double"))

    summary = df.groupBy("Region", "Category").agg(
        spark_sum("Sales").alias("total_sales"),
        spark_sum("Profit").alias("total_profit"),
        count("*").alias("order_count")
    ).orderBy(col("total_sales").desc())

    return df, summary

clean_df, summary_df = process_superstore_data(csv_path)
print("pipeline finished")
print("final row count:", clean_df.count())
summary_df.show(10)

In [ ]:
spark.stop()